# Tech Note - FFT

FFTを使うための必要事項をこの技術ノートに記す。<br>
特に、周波数分解能について、参考書やWeb記事を引用しながら整理する。

# 1. FFTの周波数分解能は、複素フーリエ級数の時間窓長で決まる

## 1.1 複素指数関数 (オイラーの公式)
波形を数学的に扱いやすくするために、一般に複素指数関数が使われる。実数 $\theta$ と ネイピア数 $e$ = 2.71828 を用いて：

$$
e^{i\theta} = \cos \theta + i \sin \theta \quad \quad (1.1)
$$

## 1.2 複素フーリエ級数
複素指数関数を使うと、時間窓 $0 \le t < T$ で "連続的な" 信号 $x(t)$ は複素フーリエ級数で次のように表される：

$$
\begin{align}
x(t) &= \cdots + c_{-2} e^{i \frac{2 \pi}{T} \times (-2t)} + c_{-1} e^{i \frac{2 \pi}{T} \times (-t)} \notag \\
&+ c_0 \times 1 \notag \\
&+ c_{1} e^{i \frac{2 \pi}{T} \times t} + c_{2} e^{i \frac{2 \pi}{T} \times 2t} + \cdots \notag \quad \quad (1.2)
\end{align}
$$

(参考：道具としてのフーリエ解析、涌井良幸・涌井貞美、日本実業出版社、2014年)

各振動数 $\omega$ [rad/s] と 周波数 $f$ [Hz] の関係 $\omega = 2 \pi f$ を念頭に置いて、式(1.2)の指数部に注目する。

$ \cdots $ <br>
$ i \frac{2 \pi}{T} \times (-2t) $ <br>
$ i \frac{2 \pi}{T} \times (-t) $ <br>
$ i \frac{2 \pi}{T} \times 0t $ <br>
$ i \frac{2 \pi}{T} \times t $ <br>
$ i \frac{2 \pi}{T} \times 2t $ <br>
$ \cdots $

複素フーリエ級数は、時間窓長 $T$ の逆数 $1/T$ を基準として、その整数倍の周波数成分の和で構成されている。

## 1.3 複素フーリエ級数の周波数分解能

これより、時間窓 $0 \le t < T$ で "連続的な" 信号 $x(t)$ 複素フーリエ級数で表す場合、周波数分解能 $\Delta f$ は時間窓長 $T$ によって決まることが分かる。
$$
\Delta f = \frac{1}{T} \quad \quad (1.3)
$$

**注意#1** <br>
これはDFTやFFTを適用する離散信号に対するものではなく、連続波形を複素フーリエ展開するときの理論値である。<br>
離散信号のサンプリング周波数（後述）と混同しないよう、はっきりと区別すること。

**注意#2** <br>
周波数は "数値が小さいほど"、"ゆっくりとした（変化ペースが遅い）" 波形を表す。
* 正：周波数の分解能が"高い" --> より"遅い・ゆっくりとした" 変化を表現できる<br>
* 誤：周波数の分解能が"高い" --> より"細かい・高速な"変化を表現できる 

（イメージ）
* 周波数分解能が高い --> "なめらか・高精細な" スローモーション再生
* 周波数分解能が低い --> "ガクガク・粗い" スローモーション再生

In [14]:
# 例：時間窓長T=1秒のとき
T = 1
f_base = 1 / T
print(f"時間窓長 T={T} 秒")
print(f"基準周波数 {f_base=} Hz")
print(f"x2 周波数 {2*f_base=} Hz")
print(f"x3 周波数 {3*f_base=} Hz")

時間窓長 T=1 秒
基準周波数 f_base=1.0 Hz
x2 周波数 2*f_base=2.0 Hz
x3 周波数 3*f_base=3.0 Hz


In [15]:
# 例：時間窓長T=10秒のとき
T = 10
f_base = 1 / T
print(f"時間窓長 T={T} 秒")
print(f"基準周波数 {f_base=} Hz")
print(f"x2 周波数 {2*f_base=} Hz")
print(f"x3 周波数 {3*f_base=} Hz")

時間窓長 T=10 秒
基準周波数 f_base=0.1 Hz
x2 周波数 2*f_base=0.2 Hz
x3 周波数 3*f_base=0.30000000000000004 Hz


# 2. 離散信号と周波数分解能

## 2.1 離散信号：時間窓長とサンプリング点数との関係

実際に処理する信号は離散値である。離散信号における時間窓長$T$を確認する。

サンプリング点数 $N$・・・FFTの時間窓での有限点数

FFTの時間窓長 $T$ は、サンプリング点数 $N$ とサンプリング周期 $f_s$（サンプリング間隔 $\Delta t$）で決まる。

* $N$：サンプリング点数
* $\Delta t$ [s] ：サンプリング間隔
* $f_s$ [Hz]：サンプリング周波数
* $T$ [s]：時間窓長

$$
T = N \Delta t = \frac{N}{f_s} \quad \quad (2.1)
$$

例：
* $N$ = 2048
* $f_s$ = 51.2 [kHz]

$$
T = \frac{2048}{51.2 \times 1000} = 0.04 [\mathrm{s}] = 40 [\mathrm{ms}]
$$

引用：小野測器 時間窓長とサンプリング点数との関係は？<br>
https://www.onosokki.co.jp/HP-WK/c_support/faq/fft_common/fft_analys_3.htm

In [7]:
2048 / (51.2 * 1000)

0.04

## 2.2 離散信号：周波数分解能

複素フーリエ級数の周波数分解能の式 (式1.3)に、離散信号の式 (式2.1) を代入して、離散信号における周波数分解能を得る：

* $\Delta f$ [Hz]：周波数分解能

$$
\Delta f = \frac{1}{T} = \frac{1}{N \Delta t} = \frac{f_s}{N} \quad \quad (2.2)
$$

周波数分解能を高める（$\Delta f$ を小さくする）には、時間窓長 $T$ を大きくする必要がある。そのためには、
* サンプリング周波数 $f_s$ を小さくする
* サンプリング点数 $N$ を大きくする

通常は、分析周波数レンジを決めると必然的にサンプリング周波数が自動的に決定される。<br>
そのため現実的には、周波数分解能は主としてサンプリング点数 $N$ に依存する。

引用：小野測器 周波数分解能はどのように決めるのか？<br>
https://www.onosokki.co.jp/HP-WK/c_support/faq/fft_common/fft_analys_4.htm

## 2.3 離散信号：高周波信号への対応能力

ナイキスト周波数 $f_n$ は、サンプリング周波数の1/2の周波数：
$$
f_n = \frac{f_s}{2} \quad \quad (2.3)
$$

ナイキスト周波数 $f_n$ より高い周波数の信号をサンプリングした場合、折り返し（エイリアシング）が生じ、再生時に、元信号を忠実に再現できない。

高周波信号を正しく分析するには、サンプリング周波数を高める必要がある。<br>

（イメージ）
* サンプリング周波数が高い --> 元信号の"完全再現"
* サンプリング周波数が低い --> 再生時に"ノイズ混入"

引用：Wikipedia ナイキスト周波数<br>
https://ja.wikipedia.org/wiki/%E3%83%8A%E3%82%A4%E3%82%AD%E3%82%B9%E3%83%88%E5%91%A8%E6%B3%A2%E6%95%B0

In [20]:
# 例：サンプリング期間=10秒、サンプリング周波数=1Hz
fs = 1  # Hz
ds = 1/fs  # 秒
f_nyquest = fs / 2  # ナイキスト周波数
print(f"サンプリング周波数 fs={fs} Hz, サンプリング間隔 ds={ds} 秒, ナイキスト周波数 f_nyquest={f_nyquest} Hz")
N = 10  # 点
print(f"10秒間サンプリング数 N={N} 点")
T = N / fs  # 秒
df = 1 / T  # Hz
print(f"時間窓長 T={T} 秒, 周波数分解能 df={df} Hz")

サンプリング周波数 fs=1 Hz, サンプリング間隔 ds=1.0 秒, ナイキスト周波数 f_nyquest=0.5 Hz
10秒間サンプリング数 N=10 点
時間窓長 T=10.0 秒, 周波数分解能 df=0.1 Hz


In [21]:
# 例：サンプリング期間=10秒、サンプリング周波数=10Hz
fs = 10  # Hz
ds = 1/fs  # 秒
f_nyquest = fs / 2  # ナイキスト周波数
print(f"サンプリング周波数 fs={fs} Hz, サンプリング間隔 ds={ds} 秒, ナイキスト周波数 f_nyquest={f_nyquest} Hz")
N = 100  # 点
print(f"10秒間サンプリング数 N={N} 点")
T = N / fs  # 秒
df = 1 / T  # Hz
print(f"時間窓長 T={T} 秒, 周波数分解能 df={df} Hz")

サンプリング周波数 fs=10 Hz, サンプリング間隔 ds=0.1 秒, ナイキスト周波数 f_nyquest=5.0 Hz
10秒間サンプリング数 N=100 点
時間窓長 T=10.0 秒, 周波数分解能 df=0.1 Hz


# 3. 周波数について、明確に区別すべきこと

「周波数分解能」と「高周波への対応能力」は、混同しがちだが、明確に区別する必要がある。

## 3.1 周波数分解能（複素フーリエ級数の理論値）
時間窓長が長い --> 周波数分解能が高い （低周波数成分の "なめらか・高精細" なスローモーション再生）

## 3.2 高周波への対応能力（サンプリングの細かさ）
高速サンプリング --> 高周波成分の分析が可能（元信号の完全再現）